<a href="https://colab.research.google.com/github/mobeena2025-tech/NGO123/blob/main/final_fml_project_finplot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn plotly ipywidgets shap

import IPython
IPython.display.clear_output()

print("All packages ready!")

All packages ready!


In [ ]:
from google.colab import files
import pandas as pd
import io

uploaded = files.upload()

csv_file = [f for f in uploaded.keys() if f.endswith(".csv")][0]

raw_df = pd.read_csv(io.BytesIO(uploaded[csv_file]))

print(f"Loaded: {csv_file}")
print(raw_df.shape)

raw_df.head()

Saving personal_finance_dataset.csv to personal_finance_dataset.csv
Loaded: personal_finance_dataset.csv
(100, 14)


,Income,Rent,Groceries,Utilities,Transportation,Healthcare,Education,Dining,Entertainment,Insurance,Investments,Loan Payments,Emergency Fund,Credit Score
0,197621,34043,481,142,214,121,209,139,332,532,10823,191,223,699
1,162475,43934,774,130,466,382,1436,479,142,559,22485,1657,890,583
2,71853,20811,548,151,179,160,689,154,77,489,4635,1735,352,715
3,41390,12249,749,111,598,243,161,350,351,733,7567,740,591,678
4,48233,7677,433,277,248,90,476,153,224,384,5692,1708,373,663


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

import warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, r2_score

print("Imports loaded")

Imports loaded


In [ ]:
col_map = {
    "Income": "income",
    "Rent": "rent",
    "Groceries": "groceries",
    "Utilities": "utilities",
    "Transportation": "transport",
    "Healthcare": "healthcare",
    "Education": "education",
    "Dining": "dining",
    "Entertainment": "entertainment",
    "Insurance": "insurance",
    "Investments": "investments",
    "Loan Payments": "loan_payment",
    "Emergency Fund": "emergency_fund",
    "Credit Score": "credit_score"
}

df = raw_df.rename(columns=col_map).copy()

df["subscriptions"] = 0
df["income_type"] = 0
df["scenario"] = 0
df["long_term_goal"] = 0

df["essential_spending"] = (
    df["rent"] + df["groceries"] + df["utilities"] +
    df["transport"] + df["healthcare"] + df["insurance"]
)

df["disc_spending"] = (
    df["dining"] + df["entertainment"] +
    df["subscriptions"] + df["education"]
)

df["total_expense"] = (
    df["essential_spending"] +
    df["disc_spending"] +
    df["loan_payment"]
)

df["expense_ratio"] = df["total_expense"] / df["income"]
df["invest_ratio"] = df["investments"] / df["income"]

df["net_savings"] = (
    df["income"] -
    df["total_expense"] -
    df["investments"]
)

df["savings_rate"] = (
    df["net_savings"] / df["income"] * 100
).clip(0, 80)

df["dti"] = df["loan_payment"] / df["income"]

# Target columns

df["stress"] = np.where(
    (df["expense_ratio"] > 0.7) &
    (df["credit_score"] < 650),
    2,
    np.where(df["expense_ratio"] > 0.5, 1, 0)
)

df["cash_flow"] = np.where(
    df["net_savings"] > 1000,
    2,
    np.where(df["net_savings"] > 0, 1, 0)
)

df["budget_goal"] = df["income"] * 0.20

df["savings_goal_met"] = np.where(
    df["net_savings"] >= df["budget_goal"],
    1,
    0
)

df["actual_savings"] = df["net_savings"].clip(0)

print("Preprocessing completed")
df.head()

Preprocessing completed


,income,rent,groceries,utilities,transport,healthcare,education,dining,entertainment,insurance,...,expense_ratio,invest_ratio,net_savings,savings_rate,dti,stress,cash_flow,budget_goal,savings_goal_met,actual_savings
0,197621,34043,481,142,214,121,209,139,332,532,...,0.184211,0.054766,150394,76.102236,0.000966,0,2,39524.2,1,150394
1,162475,43934,774,130,466,382,1436,479,142,559,...,0.307487,0.138391,90031,55.412217,0.010198,0,2,32495.0,1,90031
2,71853,20811,548,151,179,160,689,154,77,489,...,0.347835,0.064507,42225,58.765814,0.024147,0,2,14370.6,1,42225
3,41390,12249,749,111,598,243,161,350,351,733,...,0.393453,0.182822,17538,42.372554,0.017879,0,2,8278.0,1,17538
4,48233,7677,433,277,248,90,476,153,224,384,...,0.241951,0.118010,30871,64.003898,0.035411,0,2,9646.6,1,30871


In [ ]:
FEATURES = [
    "income","rent","groceries","utilities","transport","healthcare",
    "education","dining","entertainment","insurance","subscriptions",
    "investments","loan_payment","emergency_fund","credit_score",
    "income_type","scenario","essential_spending","disc_spending",
    "total_expense","expense_ratio","invest_ratio","net_savings",
    "savings_rate","dti","long_term_goal"
]

# Model 1
X1, y1 = df[FEATURES], df["stress"]
X1tr, X1te, y1tr, y1te = train_test_split(
    X1, y1, test_size=0.25, random_state=42, stratify=y1
)

rf_stress = RandomForestClassifier(n_estimators=300, random_state=42)
rf_stress.fit(X1tr, y1tr)
acc1 = accuracy_score(y1te, rf_stress.predict(X1te))

# Model 2
X2, y2 = df[FEATURES], df["savings_goal_met"]
X2tr, X2te, y2tr, y2te = train_test_split(
    X2, y2, test_size=0.25, random_state=42
)

dt_goal = DecisionTreeClassifier(max_depth=6, random_state=42)
dt_goal.fit(X2tr, y2tr)
acc2 = accuracy_score(y2te, dt_goal.predict(X2te))

# Model 3
X3, y3 = df[FEATURES], df["actual_savings"]
X3tr, X3te, y3tr, y3te = train_test_split(
    X3, y3, test_size=0.25, random_state=42
)

rf_savings = RandomForestRegressor(n_estimators=300, random_state=42)
rf_savings.fit(X3tr, y3tr)
r2_3 = r2_score(y3te, rf_savings.predict(X3te))

# Model 4
X4, y4 = df[FEATURES], df["cash_flow"]
X4tr, X4te, y4tr, y4te = train_test_split(
    X4, y4, test_size=0.25, random_state=42
)

rf_cashflow = RandomForestClassifier(n_estimators=200, random_state=42)
rf_cashflow.fit(X4tr, y4tr)
acc4 = accuracy_score(y4te, rf_cashflow.predict(X4te))

print("Stress Accuracy:", acc1)
print("Goal Accuracy:", acc2)
print("Savings R2:", r2_3)
print("Cashflow Accuracy:", acc4)

Stress Accuracy: 1.0
Goal Accuracy: 1.0
Savings R2: 0.9820534027364671
Cashflow Accuracy: 1.0


In [ ]:
def get_recommendations(d):
    recommendations = []

    if d["expense_ratio"] > 0.80:
        recommendations.append("Reduce unnecessary expenses like dining and entertainment.")

    if d["savings_rate"] < 15:
        recommendations.append("Try to save at least 20% of your monthly income.")

    if d["dti"] > 0.40:
        recommendations.append("Reduce loan burden and avoid taking new loans.")

    if d["invest_ratio"] < 0.10:
        recommendations.append("Increase monthly investments for better long-term growth.")

    if d["credit_score"] < 700:
        recommendations.append("Improve your credit score by paying bills on time.")

    if d["emergency_fund"] < d["total_expense"] * 3:
        recommendations.append("Build at least 3–6 months emergency fund.")

    if len(recommendations) == 0:
        recommendations.append("Excellent financial health! Keep maintaining this discipline.")

    return recommendations

print("Recommendation engine ready")

Recommendation engine ready


In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

sample = df.iloc[0]

display(HTML("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;600;700&display=swap');

* { font-family: 'Poppins', sans-serif; }

.hero {
    background: linear-gradient(135deg,#1e293b,#312e81);
    padding: 30px;
    border-radius: 20px;
    color: white;
    margin-bottom: 20px;
}

.section {
    background: white;
    padding: 18px;
    border-radius: 14px;
    margin: 15px 0;
    border-left: 5px solid #6366f1;
    box-shadow: 0 6px 16px rgba(0,0,0,0.06);
}

.section h3 {
    margin: 0 0 10px;
}
</style>

<div class="hero">
    <h1>◈ Smart Finance Dashboard (₹ INR)</h1>
    <p>Interactive ML-powered Financial Analysis System</p>
</div>
"""))

style = {"description_width": "160px"}
layout = widgets.Layout(width="320px")

def box(desc, val):
    return widgets.BoundedFloatText(
        value=float(val),
        min=0,
        max=999999999,
        step=500,
        description=desc,
        style=style,
        layout=layout
    )

# Create widgets (₹ added)
w_income = box("Income (₹)", sample["income"])
w_rent = box("Rent (₹)", sample["rent"])
w_groceries = box("Groceries (₹)", sample["groceries"])
w_utilities = box("Utilities (₹)", sample["utilities"])
w_transport = box("Transport (₹)", sample["transport"])
w_healthcare = box("Healthcare (₹)", sample["healthcare"])
w_education = box("Education (₹)", sample["education"])
w_dining = box("Dining (₹)", sample["dining"])
w_entertainment = box("Entertainment (₹)", sample["entertainment"])
w_insurance = box("Insurance (₹)", sample["insurance"])
w_investments = box("Investments (₹)", sample["investments"])
w_loan = box("Loan (₹)", sample["loan_payment"])
w_emergency = box("Emergency Fund (₹)", sample["emergency_fund"])
w_credit = box("Credit Score", sample["credit_score"])

# Sections
display(HTML('<div class="section"><h3>🏦 Income</h3></div>'))
display(w_income)

display(HTML('<div class="section"><h3>🏠 Fixed Expenses</h3></div>'))
display(widgets.HBox([w_rent, w_groceries]))
display(widgets.HBox([w_utilities, w_transport]))
display(widgets.HBox([w_healthcare, w_insurance]))

display(HTML('<div class="section"><h3>🎯 Lifestyle Expenses</h3></div>'))
display(widgets.HBox([w_dining, w_entertainment]))
display(widgets.HBox([w_education]))

display(HTML('<div class="section"><h3>📈 Financial Info</h3></div>'))
display(widgets.HBox([w_investments, w_loan]))
display(widgets.HBox([w_emergency, w_credit]))


BoundedFloatText(value=197621.0, description='Income (₹)', layout=Layout(width='320px'), max=999999999.0, step…

In [ ]:
import plotly.graph_objects as go
from IPython.display import display, HTML

# User input values
income = w_income.value
rent = w_rent.value
groceries = w_groceries.value
utilities = w_utilities.value
transport = w_transport.value
healthcare = w_healthcare.value
education = w_education.value
dining = w_dining.value
entertainment = w_entertainment.value
insurance = w_insurance.value
investments = w_investments.value
loan_payment = w_loan.value
emergency_fund = w_emergency.value
credit_score = w_credit.value

subscriptions = 0
income_type = 0
scenario = 0
long_term_goal = 0

# Calculations
essential_spending = (
    rent + groceries + utilities +
    transport + healthcare + insurance
)

disc_spending = (
    dining + entertainment +
    subscriptions + education
)

total_expense = (
    essential_spending +
    disc_spending +
    loan_payment
)

expense_ratio = total_expense / income
invest_ratio = investments / income
net_savings = income - total_expense - investments
savings_rate = max(0, (net_savings / income) * 100)
dti = loan_payment / income

# Prediction row
user_row = pd.DataFrame([{
    "income": income,
    "rent": rent,
    "groceries": groceries,
    "utilities": utilities,
    "transport": transport,
    "healthcare": healthcare,
    "education": education,
    "dining": dining,
    "entertainment": entertainment,
    "insurance": insurance,
    "subscriptions": subscriptions,
    "investments": investments,
    "loan_payment": loan_payment,
    "emergency_fund": emergency_fund,
    "credit_score": credit_score,
    "income_type": income_type,
    "scenario": scenario,
    "essential_spending": essential_spending,
    "disc_spending": disc_spending,
    "total_expense": total_expense,
    "expense_ratio": expense_ratio,
    "invest_ratio": invest_ratio,
    "net_savings": net_savings,
    "savings_rate": savings_rate,
    "dti": dti,
    "long_term_goal": long_term_goal
}])

# ML Predictions
stress = rf_stress.predict(user_row)[0]
goal = dt_goal.predict(user_row)[0]
future_savings = rf_savings.predict(user_row)[0]
cashflow = rf_cashflow.predict(user_row)[0]

stress_map = {
    0: "Low 😊",
    1: "Medium ⚠️",
    2: "High 🔴"
}

cash_map = {
    0: "Negative 📉",
    1: "Neutral ➖",
    2: "Positive 📈"
}

goal_text = "Achievable ✅" if goal == 1 else "Not Achievable ❌"

# HTML Dashboard (₹ Applied)
display(HTML(f"""
<style>
.result-box {{
    background: linear-gradient(135deg, #111827, #1e293b);
    padding: 30px;
    border-radius: 22px;
    color: white;
    margin-top: 25px;
}}

.metric {{
    background: white;
    color: #111827;
    padding: 18px;
    border-radius: 16px;
    margin: 12px 0;
}}

.metric h3 {{
    margin: 0;
    color: #4f46e5;
}}

.metric p {{
    font-size: 20px;
    font-weight: 700;
}}
</style>

<div class="result-box">
    <h1>📊 Smart Financial Dashboard (₹ INR)</h1>
</div>

<div class="metric">
    <h3>Total Expense</h3>
    <p>₹{total_expense:,.2f}</p>
</div>

<div class="metric">
    <h3>Net Savings</h3>
    <p>₹{net_savings:,.2f}</p>
</div>

<div class="metric">
    <h3>Savings Rate</h3>
    <p>{savings_rate:.2f}%</p>
</div>

<div class="metric">
    <h3>Stress Level</h3>
    <p>{stress_map[stress]}</p>
</div>

<div class="metric">
    <h3>Savings Goal</h3>
    <p>{goal_text}</p>
</div>

<div class="metric">
    <h3>Predicted Future Savings</h3>
    <p>₹{future_savings:,.2f}</p>
</div>

<div class="metric">
    <h3>Cash Flow Status</h3>
    <p>{cash_map[cashflow]}</p>
</div>
"""))

# PIE CHART
fig1 = go.Figure(data=[go.Pie(
    labels=["Rent","Groceries","Utilities","Transport",
            "Healthcare","Education","Dining","Entertainment","Insurance","Loan"],
    values=[rent, groceries, utilities, transport,
            healthcare, education, dining, entertainment, insurance, loan_payment],
    hole=0.4
)])
fig1.update_layout(title="Expense Breakdown (₹)")
fig1.show()

# BAR CHART
fig2 = go.Figure()
fig2.add_trace(go.Bar(
    x=["Income","Expense","Savings"],
    y=[income, total_expense, net_savings]
))
fig2.update_layout(title="Income vs Expense vs Savings (₹)")
fig2.show()

# GAUGE
fig3 = go.Figure(go.Indicator(
    mode="gauge+number",
    value=savings_rate,
    title={'text': "Savings Rate %"},
    gauge={'axis': {'range': [0, 100]}}
))
fig3.show()

print("Dashboard Generated in ₹ Successfully")

Dashboard Generated in ₹ Successfully


In [ ]:
from IPython.display import display, HTML

user = user_row.iloc[0]

new_income = user["income"] * 1.20
new_expense = user["total_expense"] * 0.85

new_savings = new_income - new_expense
new_rate = (new_savings / new_income) * 100

display(HTML(f"""
<style>
.box {{
    background: linear-gradient(135deg,#111827,#1e293b);
    padding: 25px;
    border-radius: 18px;
    color: white;
    margin-top: 20px;
}}

.metric {{
    background: white;
    color: black;
    padding: 15px;
    border-radius: 12px;
    margin: 10px 0;
}}
</style>

<div class="box">
    <h2>🔮 What-if Scenario Analysis (₹ INR)</h2>
</div>

<div class="metric">
    <b>Old Savings Rate:</b> {user['savings_rate']:.2f}%
</div>

<div class="metric">
    <b>New Savings Rate:</b> {new_rate:.2f}%
</div>

<div class="metric">
    <b>New Income:</b> ₹{new_income:,.0f}
</div>

<div class="metric">
    <b>New Expense:</b> ₹{new_expense:,.0f}
</div>

<div class="metric">
    <b>New Savings:</b> ₹{new_savings:,.0f}
</div>
"""))

In [ ]:
final_report = df.copy()

final_report.to_csv("final_finance_report.csv", index=False)

print("Final report exported successfully")
print("File name: final_finance_report.csv")

Final report exported successfully
File name: final_finance_report.csv
